In [ ]:
import pandas as pd
import glob
import os

master_path = "/Users/EthanMcElhone/Desktop/Master.xlsx"
borough_folder = "/Users/EthanMcElhone/Desktop/Borough"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Borough.xlsx"

master_df = pd.read_excel(master_path, dtype=str)

master_df.rename(columns={master_df.columns[5]: "transactionid"}, inplace=True)
master_df["transactionid"] = master_df["transactionid"].astype(str)

csv_files = glob.glob(os.path.join(borough_folder, "*.csv")) + glob.glob(os.path.join(borough_folder, "*.CSV"))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in folder: {borough_folder}")

borough_mapping = []

for file in csv_files:
    df = pd.read_csv(file, dtype=str)

    df.rename(columns={df.columns[8]: "transactionid"}, inplace=True)
    df["transactionid"] = df["transactionid"].astype(str)

    borough_name = os.path.basename(file).split("_link_")[0]
    borough_name = os.path.splitext(borough_name)[0]  

    df_borough = pd.DataFrame({
        "transactionid": df["transactionid"],
        "borough": borough_name
    })

    borough_mapping.append(df_borough)

borough_df = pd.concat(borough_mapping, ignore_index=True)

merged_df = master_df.merge(borough_df, on="transactionid", how="left")

merged_df.to_excel(output_file, index=False)
print(f"Master file updated with borough column! Saved as: {output_file}")


Master file updated with borough column! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Borough.xlsx


In [ ]:
import pandas as pd
import os

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
cpih_file = "/Users/EthanMcElhone/Downloads/CPIH_inflation_statistics.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_CPIH_Lagged.xlsx"

master_df = pd.read_excel(master_file, dtype=str)

if 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must have a 'dateoftransfer' column.")

master_df['dateoftransfer'] = pd.to_datetime(master_df['dateoftransfer'], errors='coerce')

master_df['year'] = master_df['dateoftransfer'].dt.year.astype(int)
master_df['month'] = master_df['dateoftransfer'].dt.month

def month_to_quarter(month):
    if month in [1,2,3]: return "Q1"
    elif month in [4,5,6]: return "Q2"
    elif month in [7,8,9]: return "Q3"
    elif month in [10,11,12]: return "Q4"
    else: return None

master_df['quarter'] = master_df['month'].apply(month_to_quarter)

master_df['lagged_year'] = master_df['year'] - 1
master_df['lagged_year_quarter'] = master_df['lagged_year'].astype(str) + " " + master_df['quarter']

cpih_df = pd.read_csv(cpih_file, dtype=str)
cpih_df.rename(columns={'year':'year_quarter'}, inplace=True)  # year column contains "YYYY QX"

merged_df = master_df.merge(
    cpih_df[['year_quarter','cpih_rate']], 
    left_on='lagged_year_quarter', right_on='year_quarter', how='left'
)

merged_df = merged_df.drop(columns=['year_quarter','year','month','quarter','lagged_year','lagged_year_quarter'])
merged_df.rename(columns={'cpih_rate':'cpih_rate_lagged_1yr'}, inplace=True)

merged_df['dateoftransfer'] = merged_df['dateoftransfer'].dt.strftime('%Y-%m-%d')

merged_df.to_excel(output_file, index=False)
print(f"Master file updated with 1-year lagged CPIH rate! Saved as: {output_file}")


/var/folders/x9/z2rcxv3j2yb764n0kvtclgsr0000gn/T/ipykernel_99629/265565432.py:17: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  master_df['dateoftransfer'] = pd.to_datetime(master_df['dateoftransfer'], errors='coerce')


Master file updated with 1-year lagged CPIH rate! Saved as: /Users/EthanMcElhone/Desktop/Master_with_CPIH_Lagged.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
master_df = pd.read_excel(master_file, dtype=str)

if 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must have a 'dateoftransfer' column.")

master_df['dateoftransfer'] = pd.to_datetime(master_df['dateoftransfer'], errors='coerce')

master_df['year'] = master_df['dateoftransfer'].dt.year.astype(str)

master_df['dateoftransfer'] = master_df['dateoftransfer'].dt.strftime('%Y-%m-%d')

output_file = "/Users/EthanMcElhone/Desktop/Master_with_Year.xlsx"
master_df.to_excel(output_file, index=False)
print(f"Master file updated with 'year' column and cleaned date! Saved as: {output_file}")


Master file updated with 'year' column and cleaned date! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Year.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
crime_file = "/Users/EthanMcElhone/Desktop/crime.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Crime.xlsx"

master_df = pd.read_excel(master_file, dtype=str)

if 'borough' not in master_df.columns or 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must have 'borough' and 'dateoftransfer' columns.")

master_df['year'] = master_df['dateoftransfer'].str[:4]  

crime_df = pd.read_csv(crime_file, dtype=str)

master_df['borough'] = master_df['borough'].str.strip().str.lower()
crime_df['borough'] = crime_df['borough'].str.strip().str.lower()

if not {'borough','year','crime'}.issubset(crime_df.columns):
    raise ValueError("Crime CSV must have 'borough', 'year', and 'crime' columns.")

merged_df = master_df.merge(
    crime_df[['borough', 'year', 'crime']],
    on=['borough', 'year'],
    how='left'
)

merged_df.rename(columns={'crime': 'crime_same_year'}, inplace=True)

merged_df.to_excel(output_file, index=False)
print(f"Master file updated with same-year crime rate! Saved as: {output_file}")


Master file updated with same-year crime rate! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Crime.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
ptal_file = "/Users/EthanMcElhone/Downloads/PTAL.csv"  
output_file = "/Users/EthanMcElhone/Desktop/Master_with_PTAL.xlsx"

master_df = pd.read_excel(master_file, dtype=str)

if 'postcode' not in master_df.columns:
    raise ValueError("Master file must contain a 'postcode' column.")

ptal_df = pd.read_csv(ptal_file, dtype=str)

if not {'postcode', 'ptal'}.issubset(ptal_df.columns):
    raise ValueError("PTAL file must contain 'postcode' and 'ptal' columns.")

master_df['postcode_clean'] = master_df['postcode'].str.replace(" ", "").str.lower()
ptal_df['postcode_clean'] = ptal_df['postcode'].str.replace(" ", "").str.lower()

merged_df = master_df.merge(
    ptal_df[['postcode_clean', 'ptal']],
    on='postcode_clean',
    how='left'
)

merged_df.drop(columns=['postcode_clean'], inplace=True)

merged_df.to_excel(output_file, index=False)

print(f"PTAL merged successfully! Saved to: {output_file}")


PTAL merged successfully! Saved to: /Users/EthanMcElhone/Desktop/Master_with_PTAL.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
unemployment_file = "/Users/EthanMcElhone/Downloads/Unemployment rate.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Lagged_Unemployment.xlsx"

master_df = pd.read_excel(master_file, dtype=str)

if 'year' not in master_df.columns:
    raise ValueError("Master file must contain a 'year' column.")

master_df['year'] = master_df['year'].astype(int)
master_df['lagged_year'] = master_df['year'] - 1

unemp_df = pd.read_csv(unemployment_file, dtype=str)

if 'year' not in unemp_df.columns or 'unemployment_rate' not in unemp_df.columns:
    raise ValueError("Unemployment CSV must contain 'year' and 'unemployment_rate' columns.")

unemp_df['year'] = unemp_df['year'].astype(int)
unemp_df['unemployment_rate'] = unemp_df['unemployment_rate'].astype(float)

merged_df = master_df.merge(
    unemp_df[['year', 'unemployment_rate']],
    left_on='lagged_year',
    right_on='year',
    how='left'
)

merged_df = merged_df.drop(columns=['year_y'])
merged_df.rename(columns={'year_x': 'year', 'unemployment_rate': 'unemployment_rate_lagged_1yr'}, inplace=True)
merged_df = merged_df.drop(columns=['lagged_year'])

merged_df.to_excel(output_file, index=False)
print(f"Master file updated with 1-year lagged unemployment rate! Saved as: {output_file}")


Master file updated with 1-year lagged unemployment rate! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Lagged_Unemployment.xlsx


In [ ]:
import pandas as pd
import glob
import os

master_path = "/Users/EthanMcElhone/Desktop/Master.xlsx"

borough_folder = "/Users/EthanMcElhone/Desktop/Borough"

output_file = "/Users/EthanMcElhone/Desktop/master_with_borough_data.xlsx"

standard_cols = ['transactionid', 'price', 'tfarea']

if not os.path.exists(borough_folder):
    raise FileNotFoundError(f"Borough folder not found: {borough_folder}")

master_df = pd.read_excel(master_path, dtype=str)

master_df.rename(columns={master_df.columns[5]: "transactionid"}, inplace=True)
master_df["transactionid"] = master_df["transactionid"].astype(str)

csv_files = glob.glob(os.path.join(borough_folder, "*.csv")) + glob.glob(os.path.join(borough_folder, "*.CSV"))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in folder: {borough_folder}")
print(f"Found {len(csv_files)} CSV files.")

borough_data = []

for file in csv_files:
    df = pd.read_csv(file, dtype=str)

    df.rename(columns={df.columns[8]: "transactionid"}, inplace=True)
    df["transactionid"] = df["transactionid"].astype(str)

    df.columns = df.columns.str.lower().str.replace(" ", "").str.replace("\xa0", "")

    col_map = {
        "price": "price",
        "tfarea": "tfarea"
    }
    existing_map = {k: v for k, v in col_map.items() if k in df.columns}
    df.rename(columns=existing_map, inplace=True)

    available_cols = ["transactionid"] + [c for c in standard_cols if c in df.columns and c != "transactionid"]
    borough_data.append(df[available_cols])

borough_df = pd.concat(borough_data, ignore_index=True)

merged_df = master_df.merge(borough_df, on="transactionid", how="left")

merged_df.to_excel(output_file, index=False)
print(f"Merge complete! Saved as: {output_file}")


Found 34 CSV files.
Merge complete! Saved as: /Users/EthanMcElhone/Desktop/master_with_borough_data.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
summary_file = "/Users/EthanMcElhone/Desktop/property_price_summary_year_outward_areabin.xlsx"
output_file = "/Users/EthanMcElhone/Desktop/master_with_bins_and_lagged_prices.xlsx"

master = pd.read_excel(master_file, dtype=str)

required_cols = {'year', 'postcode', 'area_bin'}
if not required_cols.issubset(master.columns):
    raise ValueError(f"Master must contain: {required_cols}")

master['year'] = master['year'].astype(int)

master['outward'] = master['postcode'].str.extract(r"^([A-Za-z0-9]+)")

summary = pd.read_excel(summary_file, dtype=str)

summary['year'] = summary['year'].astype(int)
summary['average_price'] = summary['average_price'].astype(float)
summary['average_area'] = summary['average_area'].astype(float)

summary['year_lag1'] = summary['year'] + 1

summary = summary.rename(columns={
    'average_price': 'avg_price_lag1',
    'average_area': 'avg_area_lag1'
})

summary['avg_price_per_lag1'] = summary['avg_price_lag1'] / summary['avg_area_lag1']

summary_lagged = summary[[
    'year_lag1', 'outward', 'area_bin',
    'avg_price_lag1', 'avg_area_lag1', 'avg_price_per_lag1'
]]

master = master.merge(
    summary_lagged,
    how='left',
    left_on=['year', 'outward', 'area_bin'],
    right_on=['year_lag1', 'outward', 'area_bin']
)

master = master.drop(columns=['year_lag1'])

master.to_excel(output_file, index=False)

print(f"Updated master file saved: {output_file}")


Updated master file saved: /Users/EthanMcElhone/Desktop/master_with_bins_and_lagged_prices.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
mortgage_file = "/Users/EthanMcElhone/Downloads/Mortgate rate by year.csv"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Lagged_Mortgage.xlsx"

master_df = pd.read_excel(master_file, dtype=str)

if 'year' not in master_df.columns or 'dateoftransfer' not in master_df.columns:
    raise ValueError("Master file must contain 'year' and 'dateoftransfer' columns.")

master_df['year'] = master_df['year'].astype(int)

master_df['lagged_year'] = master_df['year'] - 1

mort_df = pd.read_csv(mortgage_file, dtype=str)

mort_df.rename(columns={'year': 'mort_year', 'mortgage_rate': 'mortgage_rate'}, inplace=True)

mort_df['mort_year'] = mort_df['mort_year'].astype(int)
mort_df['mortgage_rate'] = mort_df['mortgage_rate'].astype(float)

merged_df = master_df.merge(
    mort_df[['mort_year', 'mortgage_rate']],
    left_on='lagged_year',
    right_on='mort_year',
    how='left'
)

merged_df = merged_df.drop(columns=['mort_year'])
merged_df.rename(columns={'mortgage_rate': 'mortgage_rate_lagged_1yr'}, inplace=True)

merged_df = merged_df.drop(columns=['lagged_year'])

merged_df.to_excel(output_file, index=False)
print(f"Master file updated with 1-year lagged mortgage rate! Saved as: {output_file}")


Master file updated with 1-year lagged mortgage rate! Saved as: /Users/EthanMcElhone/Desktop/Master_with_Lagged_Mortgage.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
output_file = "/Users/EthanMcElhone/Desktop/property_price_summary_year_borough_outward_areabin.xlsx"

df = pd.read_excel(master_file, dtype=str)

required_cols = {'year', 'postcode', 'tfarea', 'price', 'borough', 'area_bin'}
if not required_cols.issubset(df.columns):
    raise ValueError(f"Master must include: {required_cols}")

df['year'] = df['year'].astype(int)
df['price'] = df['price'].astype(float)
df['tfarea'] = df['tfarea'].astype(float)

df['outward'] = df['postcode'].str.extract(r"^([A-Za-z0-9]+)")

summary = df.groupby(['year', 'borough', 'outward', 'area_bin']).agg(
    average_price=('price', 'mean'),
    average_area=('tfarea', 'mean'),
    property_count=('price', 'count')
).reset_index()

summary.to_excel(output_file, index=False)

print(f"Summary file created: {output_file}")

Summary file created: /Users/EthanMcElhone/Desktop/property_price_summary_year_borough_outward_areabin.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
summary_file = "/Users/EthanMcElhone/Desktop/property_price_summary_year_borough_outward_areabin.xlsx"
output_file = "/Users/EthanMcElhone/Desktop/master_with_bins_and_lagged_prices.xlsx"

master = pd.read_excel(master_file, dtype=str)

required_cols = {'year', 'postcode', 'area_bin', 'borough'}
if not required_cols.issubset(master.columns):
    raise ValueError(f"Master must contain: {required_cols}")

master['year'] = master['year'].astype(int)

master['outward'] = master['postcode'].str.extract(r"^([A-Za-z0-9]+)")

summary = pd.read_excel(summary_file, dtype=str)

summary['year'] = summary['year'].astype(int)
summary['average_price'] = summary['average_price'].astype(float)
summary['average_area'] = summary['average_area'].astype(float)

summary['year_lag1'] = summary['year'] + 1

summary = summary.rename(columns={
    'average_price': 'avg_price_lag1',
    'average_area': 'avg_area_lag1'
})

summary['avg_price_per_lag1'] = summary['avg_price_lag1'] / summary['avg_area_lag1']

summary_lagged = summary[[
    'year_lag1', 'borough', 'outward', 'area_bin',
    'avg_price_lag1', 'avg_area_lag1', 'avg_price_per_lag1'
]]

master = master.merge(
    summary_lagged,
    how='left',
    left_on=['year', 'borough', 'outward', 'area_bin'],
    right_on=['year_lag1', 'borough', 'outward', 'area_bin']
)

master = master.drop(columns=['year_lag1'])

master.to_excel(output_file, index=False)

print(f"Updated master file saved: {output_file}")


Updated master file saved: /Users/EthanMcElhone/Desktop/master_with_bins_and_lagged_prices.xlsx


In [ ]:
import pandas as pd
import glob
import os

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
borough_folder = "/Users/EthanMcElhone/Desktop/Borough"
output_file = "/Users/EthanMcElhone/Desktop/Master_with_Duration.xlsx"

master_df = pd.read_excel(master_file, dtype=str)

if 'transactionid' not in master_df.columns:
    raise ValueError("Master file must already contain 'transactionid' column.")

master_df['transactionid'] = master_df['transactionid'].astype(str)

csv_files = glob.glob(os.path.join(borough_folder, "*.csv")) + \
            glob.glob(os.path.join(borough_folder, "*.CSV"))

if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in {borough_folder}")

duration_rows = []

for file in csv_files:
    df = pd.read_csv(file, dtype=str)

    df.rename(columns={df.columns[8]: 'transactionid'}, inplace=True)
    df['transactionid'] = df['transactionid'].astype(str)

    if 'duration' not in df.columns:
        print(f"⚠️ duration column missing in {os.path.basename(file)}, skipped.")
        continue

    duration_rows.append(df[['transactionid', 'duration']])

duration_df = pd.concat(duration_rows, ignore_index=True)

duration_df = duration_df.drop_duplicates(subset='transactionid')

merged_df = master_df.merge(
    duration_df,
    on='transactionid',
    how='left'
)

merged_df.to_excel(output_file, index=False)
print(f"Duration column added successfully → {output_file}")


Duration column added successfully → /Users/EthanMcElhone/Desktop/Master_with_Duration.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
output_file = "/Users/EthanMcElhone/Desktop/outward_borough_bins_duration_type.xlsx"

df = pd.read_excel(master_file, dtype=str)

required_cols = {'postcode', 'borough', 'area_bin', 'duration', 'propertytype'}
if not required_cols.issubset(df.columns):
    raise ValueError(f"Master file must contain columns: {required_cols}")

df['outward'] = df['postcode'].str.extract(r'^([A-Za-z0-9]+)')

df_subset = df[['outward', 'borough', 'area_bin', 'duration', 'propertytype']]

df_subset = df_subset.dropna(subset=['outward', 'borough'])

final_df = df_subset.drop_duplicates().sort_values(
    by=['outward', 'borough', 'area_bin', 'duration', 'propertytype']
)

final_df.to_excel(output_file, index=False)

print(f"New spreadsheet created with unique combinations:\n{output_file}")


New spreadsheet created with unique combinations:
/Users/EthanMcElhone/Desktop/outward_borough_bins_duration_type.xlsx


In [ ]:
import pandas as pd

master_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
base_file = "/Users/EthanMcElhone/Desktop/outward_borough_bins_duration_type.xlsx"
output_file = "/Users/EthanMcElhone/Desktop/outward_borough_bins_duration_type_prices.xlsx"

master = pd.read_excel(master_file, dtype=str)
base = pd.read_excel(base_file, dtype=str)

master['outward'] = master['postcode'].str.extract(r'^([A-Za-z0-9]+)')

master['year'] = master['year'].astype(int)
master['price'] = master['price'].astype(float)

master_sub = master[
    ['outward', 'borough', 'area_bin', 'duration', 'propertytype', 'year', 'price']
]

price_panel = (
    master_sub
    .groupby(
        ['outward', 'borough', 'area_bin', 'duration', 'propertytype', 'year'],
        as_index=False
    )['price']
    .mean()
)

price_wide = price_panel.pivot(
    index=['outward', 'borough', 'area_bin', 'duration', 'propertytype'],
    columns='year',
    values='price'
)

price_wide.columns = [f"{int(col)}_price" for col in price_wide.columns]

price_wide = price_wide.reset_index()

final = base.merge(
    price_wide,
    on=['outward', 'borough', 'area_bin', 'duration', 'propertytype'],
    how='left'
)

final.to_excel(output_file, index=False)

print(f"Done! Year price columns added:\n{output_file}")


Done! Year price columns added:
/Users/EthanMcElhone/Desktop/outward_borough_bins_duration_type_prices.xlsx


In [ ]:
import pandas as pd

master_path = "/Users/EthanMcElhone/Desktop/Master.xlsx"
master_df = pd.read_excel(master_path, dtype=str)

master_2024 = master_df[master_df['year'] == '2024']

master_2024 = master_2024.rename(columns={"price": "2024_price"})

master_2024.to_excel(
    "/Users/EthanMcElhone/Desktop/Master2024.xlsx",
    index=False
)

In [ ]:
import pandas as pd

input_file = "/Users/EthanMcElhone/Desktop/Master2024.xlsx"
output_file = "/Users/EthanMcElhone/Desktop/Model2024.xlsx"

df = pd.read_excel(input_file, dtype=str)

final_df = df[[
    'propertytype',
    'duration',
    'area_bin',
    'borough',
    'postcode',
    'lat',
    'long',
    'food',
    'education',
    'culture',
    'ptal',
    'crime_lagged_1yr',
    'cpih_lagged_1yr',
    'unemployment_lagged_1yr',
    'mortgage_lagged_1yr',
    '2024_price'
]]

final_df.to_excel(output_file, index=False)

print(f"2024 modelling dataset saved to: {output_file}")


2024 modelling dataset saved to: /Users/EthanMcElhone/Desktop/Model2024.xlsx


In [ ]:
import pandas as pd

input_file = "/Users/EthanMcElhone/Desktop/Master.xlsx"
output_file = "/Users/EthanMcElhone/Desktop/ModelTrain.xlsx"

df = pd.read_excel(input_file, dtype=str)

final_df = df[[
    'propertytype',
    'duration',
    'area_bin',
    'borough',
    'postcode',
    'lat',
    'long',
    'food',
    'education',
    'culture',
    'ptal',
    'crime_lagged_1yr',
    'cpih_lagged_1yr',
    'unemployment_lagged_1yr',
    'mortgage_lagged_1yr',
    'price'
]]

final_df.to_excel(output_file, index=False)

print(f"2015-2024 modelling dataset saved to: {output_file}")


2015-2024 modelling dataset saved to: /Users/EthanMcElhone/Desktop/ModelTrain.xlsx


In [ ]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

LOW_MAX = 250_000
HIGH_MIN = 1_100_000

segments = {
    "low": (-np.inf, LOW_MAX),
    "mid": (LOW_MAX, HIGH_MIN),
    "high": (HIGH_MIN, np.inf),
}

segment_loss = {
    "low": ("huber", 0.9),
    "mid": ("regression", None),
    "high": ("huber", 0.9),
}

segment_params = {
    "low": {
        "n_estimators": 1195,
        "learning_rate": 0.03207664578812468,
        "num_leaves": 84,
        "max_depth": 13,
        "min_child_samples": 5,
        "subsample": 0.7889533873638548,
        "colsample_bytree": 0.5663454662129782,
    },
    "mid": {
        "n_estimators": 1494,
        "learning_rate": 0.0591310714508011,
        "num_leaves": 143,
        "max_depth": 19,
        "min_child_samples": 18,
        "subsample": 0.9327023224082089,
        "colsample_bytree": 0.5610102961609175,
    },
    "high": {
        "n_estimators": 1011,
        "learning_rate": 0.010555203732718746,
        "num_leaves": 130,
        "max_depth": 18,
        "min_child_samples": 12,
        "subsample": 0.8753968850598899,
        "colsample_bytree": 0.5003753045491645,
    },
}

df_2024 = pd.read_excel("/Users/EthanMcElhone/Desktop/Model2024.xlsx")
df = pd.read_excel("/Users/EthanMcElhone/Desktop/ModelTrain.xlsx")

df["price"] = df["price"].astype(float)

features = [
    'food', 'education', 'culture', 'ptal',
    'crime_lagged_1yr', 'cpih_lagged_1yr',
    'unemployment_lagged_1yr', 'mortgage_lagged_1yr',
    'propertytype', 'duration', 'area_bin',
    'borough', 'postcode', 'lat', 'long'
]

categorical_features = [
    'propertytype', 'duration', 'area_bin', 'borough', 'postcode'
]

segment_data = {}

for name, (low, high) in segments.items():
    seg_df = df[(df["price"] > low) & (df["price"] <= high)].copy()

    X = seg_df[features].copy()
    for col in categorical_features:
        X[col] = X[col].astype("category")

    y = seg_df["price"]

    segment_data[name] = {
        "X": X,
        "y": y,
        "y_log": np.log(y),
    }

segment_models = {}

for name in segments:
    loss, alpha = segment_loss[name]

    model = LGBMRegressor(
        **segment_params[name],
        objective=loss,
        random_state=42,
    )

    if loss == "huber":
        model.set_params(alpha=alpha)

    model.fit(
        segment_data[name]["X"],
        segment_data[name]["y_log"],
        categorical_feature=categorical_features
    )

    segment_models[name] = model

X_pred = df_2024[features].copy()
for col in categorical_features:
    X_pred[col] = X_pred[col].astype("category")

df_2024["2025_price"] = np.nan

for name, (low, high) in segments.items():
    mask = (df_2024["2024_price"] > low) & (df_2024["2024_price"] <= high)

    preds_log = segment_models[name].predict(X_pred.loc[mask])
    df_2024.loc[mask, "2025_price"] = np.exp(preds_log)

for name, (low, high) in segments.items():
    X_train_seg = segment_data[name]["X"]
    y_train_seg = segment_data[name]["y"]

    preds_train = np.exp(segment_models[name].predict(X_train_seg))

    b, a = np.polyfit(preds_train, y_train_seg, 1)

    mask = (df_2024["2024_price"] > low) & (df_2024["2024_price"] <= high)
    df_2024.loc[mask, "2025_price"] = (
        a + b * df_2024.loc[mask, "2025_price"]
    )

df_2024.to_excel(
    "/Users/EthanMcElhone/Desktop/Model2025.xlsx",
    index=False
)

print("Segmented, calibrated 2025 prices saved.")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002442 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8684
[LightGBM] [Info] Number of data points in the train set: 68897, number of used features: 15
[LightGBM] [Info] Start training from score 12.173818
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010336 seconds.
You can set `force_row_wise

In [ ]:
import pandas as pd

train_path = "/Users/EthanMcElhone/Desktop/ModelTrain.xlsx"
model2025_path = "/Users/EthanMcElhone/Desktop/Model2025.xlsx"
output_path = "/Users/EthanMcElhone/Desktop/ModelTrain2025.xlsx"

train_df = pd.read_excel(train_path)
df_2025 = pd.read_excel(model2025_path)

df_2025 = df_2025.drop(columns=["2024_price"], errors="ignore")

df_2025 = df_2025.rename(columns={"2025_price": "price"})

df_2025 = df_2025[train_df.columns]

train_updated = pd.concat([train_df, df_2025], ignore_index=True)

train_updated.to_excel(output_path, index=False)

print(f"Updated training set saved → {output_path}")
print(f"Rows before: {len(train_df):,}")
print(f"Rows after:  {len(train_updated):,}")


Updated training set saved → /Users/EthanMcElhone/Desktop/ModelTrain_2015_2025.xlsx
Rows before: 842,191
Rows after:  883,340


In [ ]:
import pandas as pd

train_path = "/Users/EthanMcElhone/Desktop/ModelTrain.xlsx"
model2025_path = "/Users/EthanMcElhone/Desktop/Model2025.xlsx"
output_path = "/Users/EthanMcElhone/Desktop/ModelTrain2025.xlsx"

train_df = pd.read_excel(train_path)
df_2025 = pd.read_excel(model2025_path)

df_2025 = df_2025.drop(columns=["2024_price"], errors="ignore")

df_2025 = df_2025.rename(columns={"2025_price": "price"})

df_2025["mortgage_lagged_1yr"] = 4.827
df_2025["unemployment_lagged_1yr"] = 4.11

cpih_map = {
    9.0: 3.9,
    7.7: 2.9,
    6.3: 2.9,
    4.4: 3.4,
}
df_2025["cpih_lagged_1yr"] = df_2025["cpih_lagged_1yr"].replace(cpih_map)

crime_map = {
    "barking and dagenham": 0.18,
    "barnet": 0.22,
    "bexley": 0.13,
    "brent": 0.25,
    "brent*": 0.25,
    "bromley": 0.16,
    "camden": 0.50,
    "city of london": 0.13,
    "croydon": 0.26,
    "ealing": 0.25,
    "enfield": 0.22,
    "greenwich": 0.19,
    "hackney": 0.38,
    "hammersmith and fulham": 0.31,
    "haringey": 0.28,
    "harrow": 0.15,
    "havering": 0.19,
    "hillingdon": 0.19,
    "hounslow": 0.19,
    "islington": 0.43,
    "kensington and chelsea": 0.40,
    "kingston upon thames": 0.15,
    "lambeth": 0.36,
    "lewisham": 0.19,
    "merton": 0.16,
    "newham": 0.33,
    "redbridge": 0.19,
    "richmond upon thames": 0.13,
    "southwark": 0.44,
    "sutton": 0.13,
    "tower hamlets": 0.33,
    "waltham forest": 0.20,
    "wandsworth": 0.17,
    "westminster": 0.73,
}

df_2025["crime_lagged_1yr"] = df_2025["borough"].map(crime_map)

if df_2025["crime_lagged_1yr"].isna().any():
    missing = df_2025.loc[df_2025["crime_lagged_1yr"].isna(), "borough"].unique()
    raise ValueError(f"Missing crime rates for borough(s): {missing}")

df_2025 = df_2025[train_df.columns]

train_updated = pd.concat([train_df, df_2025], ignore_index=True)

train_updated.to_excel(output_path, index=False)

print(f"Updated training set saved → {output_path}")
print(f"Rows before: {len(train_df):,}")
print(f"Rows after:  {len(train_updated):,}")


Updated training set saved → /Users/EthanMcElhone/Desktop/ModelTrain2025.xlsx
Rows before: 842,191
Rows after:  883,340


In [4]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor

LOW_MAX = 250_000
HIGH_MIN = 1_100_000

segments = {
    "low": (-np.inf, LOW_MAX),
    "mid": (LOW_MAX, HIGH_MIN),
    "high": (HIGH_MIN, np.inf),
}

segment_loss = {
    "low": ("huber", 0.9),
    "mid": ("regression", None),
    "high": ("huber", 0.9),
}

segment_params = {
    "low": {
        "n_estimators": 1195,
        "learning_rate": 0.03207664578812468,
        "num_leaves": 84,
        "max_depth": 13,
        "min_child_samples": 5,
        "subsample": 0.7889533873638548,
        "colsample_bytree": 0.5663454662129782,
    },
    "mid": {
        "n_estimators": 1494,
        "learning_rate": 0.0591310714508011,
        "num_leaves": 143,
        "max_depth": 19,
        "min_child_samples": 18,
        "subsample": 0.9327023224082089,
        "colsample_bytree": 0.5610102961609175,
    },
    "high": {
        "n_estimators": 1011,
        "learning_rate": 0.010555203732718746,
        "num_leaves": 130,
        "max_depth": 18,
        "min_child_samples": 12,
        "subsample": 0.8753968850598899,
        "colsample_bytree": 0.5003753045491645,
    },
}

df_pred = pd.read_excel("/Users/EthanMcElhone/Desktop/Model2025.xlsx")  
df_train = pd.read_excel("/Users/EthanMcElhone/Desktop/ModelTrain2025.xlsx")  

df_train["price"] = df_train["price"].astype(float)

features = [
    'food', 'education', 'culture', 'ptal',
    'crime_lagged_1yr', 'cpih_lagged_1yr',
    'unemployment_lagged_1yr', 'mortgage_lagged_1yr',
    'propertytype', 'duration', 'area_bin',
    'borough', 'postcode', 'lat', 'long'
]

categorical_features = [
    'propertytype', 'duration', 'area_bin', 'borough', 'postcode'
]

segment_data = {}

for name, (low, high) in segments.items():
    seg_df = df_train[(df_train["price"] > low) & (df_train["price"] <= high)].copy()

    X = seg_df[features].copy()
    for col in categorical_features:
        X[col] = X[col].astype("category")

    y = seg_df["price"]

    segment_data[name] = {
        "X": X,
        "y": y,
        "y_log": np.log(y),
    }

segment_models = {}

for name in segments:
    loss, alpha = segment_loss[name]

    model = LGBMRegressor(
        **segment_params[name],
        objective=loss,
        random_state=42,
        force_row_wise=True
    )

    if loss == "huber":
        model.set_params(alpha=alpha)

    model.fit(
        segment_data[name]["X"],
        segment_data[name]["y_log"],
        categorical_feature=categorical_features
    )

    segment_models[name] = model

X_pred = df_pred[features].copy()
for col in categorical_features:
    X_pred[col] = X_pred[col].astype("category")

df_pred["2026_price"] = np.nan

for name, (low, high) in segments.items():
    mask = (
        (df_pred["2025_price"] > low) &
        (df_pred["2025_price"] <= high)
    )

    preds_log = segment_models[name].predict(X_pred.loc[mask])
    df_pred.loc[mask, "2026_price"] = np.exp(preds_log)

for name in segments:
    X_train_seg = segment_data[name]["X"]
    y_train_seg = segment_data[name]["y"]

    preds_train = np.exp(segment_models[name].predict(X_train_seg))
    b, a = np.polyfit(preds_train, y_train_seg, 1)

    mask = (
        (df_pred["2025_price"] > segments[name][0]) &
        (df_pred["2025_price"] <= segments[name][1])
    )

    df_pred.loc[mask, "2026_price"] = (
        a + b * df_pred.loc[mask, "2026_price"]
    )

df_pred.to_excel(
    "/Users/EthanMcElhone/Desktop/Model2026.xlsx",
    index=False
)

print("Segmented, calibrated 2026 prices saved.")

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Total Bins 8950
[LightGBM] [Info] Number of data points in the train set: 71302, number of used features: 15
[LightGBM] [Info] Start training from score 12.173586
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Total Bins 28359
[LightGBM] [Info] Number of data points in the train set: 730916, number of used features: 15
[LightGBM] [Info] Start training from score 13.097620
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_f

In [5]:
import pandas as pd

MODEL2026_PATH = "/Users/EthanMcElhone/Desktop/Model2026.xlsx"
WEBSITE_PATH = "/Users/EthanMcElhone/Desktop/Website Data Table.xlsx"
OUTPUT_PATH = "/Users/EthanMcElhone/Desktop/WebsiteDataTable.xlsx"

model2026 = pd.read_excel(MODEL2026_PATH)
website = pd.read_excel(WEBSITE_PATH)

model2026["outward"] = model2026["postcode"].str.extract(r"^([A-Za-z0-9]+)")

key_cols = ["propertytype", "duration", "area_bin", "borough", "outward"]
price_cols = ["2025_price", "2026_price"]

model2026_unique = (
    model2026
    .sort_values(key_cols)       
    .drop_duplicates(
        subset=key_cols,
        keep="first"
    )
    [key_cols + price_cols]
)

website_updated = website.merge(
    model2026_unique,
    on=key_cols,
    how="left"
)

website_updated.to_excel(OUTPUT_PATH, index=False)

print("Website data updated with 2025 and 2026 prices.")


Website data updated with 2025 and 2026 prices.
